In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import re
from pathlib import Path
import os
import h5py
import time

from my_pkg.get_elevation_from_dsm import get_utm_3d_points_from_dsm

os.environ['PROJ_LIB'] = '/home/lty/anaconda3/envs/hloc/share/proj'

from osgeo import gdal, osr
from geopy.distance import geodesic

# from scipy.stats import pairs

from hloc import extract_features, match_features
from hloc.utils.io import get_matches, get_keypoints
from hloc.visualization import plot_images, plot_keypoints, plot_matches, read_image, add_text

from my_pkg.tools import sort_key, pixel_to_geo_coordinates, read_pairs, extract_rotation_angle, \
    draw_fusion_keyframe_traj


In [32]:
image_dir = Path("/home/lty/datasets_my/DJI/m300/")
seu_uav_dir = image_dir / "seu_uav_052409"
seu_tif_dir = image_dir / "seu_tif_m300"
output_dir = Path("/home/lty/outputs/seu0524/009")
output_dir.mkdir(exist_ok=True, parents= True)
pairs_path = output_dir/"pairs.txt"
img_list_path = output_dir/"img_list.txt"
uav_list_path = output_dir/"uav_list.txt"
tif_list_path = output_dir/"tif_list.txt"
features_path = output_dir/"features.h5"
matches_path = output_dir/"matches.h5"
loc_path = output_dir/"loc_h.txt"# 保存定位结果

In [11]:
image_extensions = ['.jpg', '.jpeg', '.png', '.tif', '.tiff']
# 收集 'seu_uav' 文件夹中的所有图像文件
uav_images = [
    str(p.relative_to(image_dir))  # 获取相对于 image_dir 的路径，包含文件夹前缀
    for p in seu_uav_dir.iterdir()
    if p.is_file() and p.suffix.lower() in image_extensions
]

# 收集 'seu_tif' 文件夹中的所有图像文件
tif_images = [
    str(p.relative_to(image_dir))  # 获取相对于 image_dir 的路径，包含文件夹前缀
    for p in seu_tif_dir.iterdir()
    if p.is_file() and p.suffix.lower() in image_extensions
]
uav_images = sorted(uav_images, key = sort_key)
tif_images = sorted(tif_images, key = sort_key)
# 将 UAV 和 TIF 图像文件列表合并
img_list = uav_images + tif_images
# 打印 img_list 以验证结果
# for img in img_list:
#     print(img)
#save img_list
with open(img_list_path, 'w') as f:
    for img in img_list:
        f.write(img + "\n")

with open(uav_list_path, 'w') as f:
    for img in uav_images:
        f.write(img + "\n")
with open(tif_list_path, 'w') as f:
    for img in tif_images:
        f.write(img + "\n")

In [12]:

t0 = time.time()
extract_features.main(
    conf=extract_features.confs['superpoint_max'],
    image_dir=image_dir,
    image_list=img_list,
    feature_path = features_path,
)
t1 = time.time()
match_features.main(
    conf=match_features.confs["superpoint+lightglue"],
    pairs=pairs_path,
    features=features_path,
    matches=matches_path,
)
t2 = time.time()
print(f"Feature extraction time: {t1-t0:.3f}s")
print(f"Feature matching time: {t2-t1:.3f}s")

[2025/06/03 14:49:14 hloc INFO] Extracting local features with configuration:
{'model': {'max_keypoints': 4096, 'name': 'superpoint', 'nms_radius': 3},
 'output': 'feats-superpoint-n4096-rmax1600',
 'preprocessing': {'grayscale': True, 'resize_force': True, 'resize_max': 1600}}
[2025/06/03 14:49:14 hloc INFO] Skipping the extraction.
[2025/06/03 14:49:14 hloc INFO] Matching local features with configuration:
{'model': {'features': 'superpoint', 'name': 'lightglue'},
 'output': 'matches-superpoint-lightglue'}
[2025/06/03 14:49:14 hloc INFO] Skipping the matching.


Feature extraction time: 0.133s
Feature matching time: 0.009s


In [13]:
geotransform = []
# 打开 geotransform.txt 文件
with open("/home/lty/scripts/seu_geotransform_fix.txt", "r") as f:
    # 逐行读取文件内容
    for line in f:
        # 去除行首尾的空白字符和换行符
        line = line.strip()
        if line:
            try:
                # 将字符串转换为浮点数
                value = float(line)
                # 将数值添加到列表中
                geotransform.append(value)
            except ValueError:
                print(f"无法将以下内容转换为数值：'{line}'")
                # 根据需要，可以选择跳过或停止程序
                continue
# 输出读取到的 geotransform 数据
print("Geotransform 数组：\n", geotransform)

Geotransform 数组：
 [668601.89603705, 0.03459999999999788, 0.0, 3548451.1491134795, 0.0, -0.03459999999998963]


In [14]:
fx, fy = 1543.468, 1543.468   # 焦距
cx, cy = 973.856, 533.855     # 主点

K = np.array([
    [fx,  0, cx],
    [ 0, fy, cy],
    [ 0,  0,  1]
], dtype=np.float32)
from my_pkg.get_elevation_from_dsm import get_utm_3d_points_from_dsm

In [15]:
# 创建源坐标系和目标坐标系
source_srs = osr.SpatialReference()
source_srs.ImportFromEPSG(32650)

target_srs = osr.SpatialReference()
target_srs.ImportFromEPSG(4326)

# 创建坐标转换对象
coord_transform = osr.CoordinateTransformation(source_srs, target_srs)



In [28]:
dsm_path = "/home/lty/data/SEU/dsm.tif"
geo_utm_path = "/home/lty/paper/results/052409/elevpnp.txt"
geo3d_path = "/home/lty/paper/results/052409/elevpnp3d.txt"
dataset = gdal.Open(dsm_path)
if dataset is None:
    raise FileNotFoundError(f"无法打开 DSM 文件：{dsm_path}")

 # 获取仿射变换和影像数据
dsm_geotransform = dataset.GetGeoTransform()
band = dataset.GetRasterBand(1)
dsm_array = band.ReadAsArray()
rows, cols = dsm_array.shape
# 提取仿射变换参数
dsm_origin_x, dsm_pixel_width, _, dsm_origin_y, _, dsm_pixel_height = dsm_geotransform

pairs = read_pairs(pairs_path)
print(f"Found {len(pairs)} image pairs.")
start_x, start_y = 0, 0
x_in_map, y_in_map = 0, 0
x_origin, y_origin = 0, 0
start = time.time()
n = 0

with open(loc_path, 'w') as loc_file, open(geo_utm_path, 'w') as geo_utm_file, open(geo3d_path, 'w') as geo3d_file:
    for img_uav, img_tif in pairs:
        print(f"UAV: {img_uav} - TIF: {img_tif}")
        tif_name = os.path.basename(img_tif)
        match = re.match(r"(\d+)_(\d+)_(\d+).tif", tif_name)
        if match:
            start_x = match.group(2)
            start_y = match.group(3)
            print(f"start_x: {start_x}, start_y: {start_y}")
            
        lu_geox = geotransform[0] + float(start_x) * geotransform[1]
        lu_geoy = geotransform[3] + float(start_y) * geotransform[5]

        print(f"lu_geox: {lu_geox}, lu_geoy: {lu_geoy}")
        
        kp1, kp2  = get_keypoints(features_path, img_uav), get_keypoints(features_path, img_tif)
        matches,scores = get_matches(matches_path, img_uav, img_tif)
        print(matches.shape)
        pts1 = kp1[matches[:,0]]
        pts2 = kp2[matches[:,1]]
        
        F, mask = cv2.findFundamentalMat(pts1, pts2, cv2.RANSAC, 1.5)
        
        if F is not None:
            inliers = mask.ravel().tolist()
            num_inliers = np.sum(inliers)
            print(f"Number of inliers: {num_inliers}")
             # 提取内点匹配对
            # 将 mask 转换为一维布尔数组
            inliers = mask.ravel().astype(bool)
            pts1_inliers = pts1[inliers]
            pts2_inliers = pts2[inliers]
            pts2_inliers_geo = np.zeros((len(pts2_inliers), 2))
            pts2_inliers_geo_utm = np.zeros((len(pts2_inliers), 2))
            print(pts2_inliers.shape)
            for idx, (x_pix, y_pix) in enumerate(pts2_inliers): 
                x_in_map = int(start_x) + x_pix
                y_in_map = int(start_y) + y_pix
                
                x_geo = geotransform[0] + x_in_map * geotransform[1]
                y_geo = geotransform[3] + y_in_map * geotransform[5]
                
                pts2_inliers_geo_utm[idx, 0] = x_geo
                pts2_inliers_geo_utm[idx, 1] = y_geo     
                x_geo = x_geo-lu_geox
                y_geo = y_geo-lu_geoy
            
                pts2_inliers_geo[idx, 0] = x_geo
                pts2_inliers_geo[idx, 1] = y_geo
                
            satellite_3d_utm = np.zeros((len(pts2_inliers_geo_utm), 3))
            for idx, (utm_x, utm_y) in enumerate(pts2_inliers_geo_utm):
                # print(f"utm_x: {utm_x}, utm_y: {utm_y}")
                pixel_x = int(round((utm_x - dsm_origin_x) / dsm_pixel_width))
                pixel_y = int(round((utm_y - dsm_origin_y) / dsm_pixel_height))
                # print(f"pixel_x: {pixel_x}, pixel_y: {pixel_y}")

                if 0 <= pixel_x < cols and 0 <= pixel_y < rows:
                    noise = np.random.normal(-2.5, 2.5)
                    elevation = dsm_array[pixel_y, pixel_x] + noise
                else:
                    elevation = -9999  # 或者使用 np.nan
                    # print("!!!!!!!!!!!!!!!!!!!!!")
                
                satellite_3d_utm[idx] = [utm_x, utm_y, elevation]
            print(f"成功构建 3D 点集，共 {len(satellite_3d_utm)} 个点")
            z_vals = satellite_3d_utm[:, 2].reshape(-1, 1)  # (N, 1)
            satellite_3d_geo = np.hstack([pts2_inliers_geo, z_vals])  
            object_points = satellite_3d_geo.astype(np.float32)
            # (N, 2) 无人机图像上的像素坐标
            image_points = pts1_inliers.astype(np.float32)
            # solvePnP：用来估计R, t
            success, rvec, tvec = cv2.solvePnP(
                objectPoints=object_points,
                imagePoints=image_points,
                cameraMatrix=K,
                distCoeffs=None,  # 无畸变时设为 None
                flags=cv2.SOLVEPNP_ITERATIVE  # 可换成 EPNP/DLS/UPnP 等
            )
            if success:
                R, _ = cv2.Rodrigues(rvec)  # 将旋转向量转换为矩阵
                camera_center = -R.T @ tvec
                # 相机 z 轴方向在世界坐标系下的方向向量
                cam_forward = R @ np.array([0, 0, 1])  # shape (3,)
                
                # 取其在地图平面 XY 上的投影
                forward_xy = cam_forward[:2]
                
                # 计算航向角（水平旋转角），以地图 x 轴为 0°
                yaw_rad = np.arctan2(forward_xy[1], forward_xy[0])
                yaw_deg = np.degrees(yaw_rad)
                
                # 保证角度在 [0, 360) 范围
                yaw_deg = (yaw_deg + 360) % 360
                print(f"Yaw angle (degrees): {yaw_deg:.2f}")
            
                # print("位姿估计成功！")
                # print("旋转矩阵 R:\n", R)
                # print("平移向量 t:\n", tvec.ravel())
                print("相机中心（世界坐标系）:\n", camera_center.ravel())
                camera_center_geo = camera_center.ravel() + np.array([lu_geox, lu_geoy, 0])
                camerax_geo = camera_center_geo[0]
                cameray_geo = camera_center_geo[1]
                cameraz_geo = camera_center_geo[2]
                camerax_geo = float(camerax_geo)
                cameray_geo = float(cameray_geo)
                cameraz_geo = float(cameraz_geo)
                # 转换为经纬度坐标（EPSG:target_epsg）
               
                lat, lon, _ = coord_transform.TransformPoint(camerax_geo, cameray_geo, cameraz_geo)
                angle = 0
                if n==0:
                    x_origin, y_origin = camerax_geo, cameray_geo
                print(f"相机中心utm:：{cameray_geo}, {camerax_geo}")
                
                x_in_map  = (camerax_geo-geotransform[0])/geotransform[1]
                y_in_map  = (cameray_geo-geotransform[3])/geotransform[5]
                geo_utm_file.write(f"{camerax_geo:.8f} {cameray_geo:.8f}\n")
                geo3d_file.write(f"{camerax_geo:.8f} {cameray_geo:.8f} {cameraz_geo:.8f}\n")
                loc_file.write(f"{img_uav} {lon:.8f} {lat:.8f} {x_in_map:.8f} {y_in_map:.8f} {float(camerax_geo):.8f} {float(cameray_geo):.8f} {camerax_geo-x_origin:.10f} {y_origin-cameray_geo:.10f} {angle:.8f}\n")
            else:
                print("solvePnP 失败，可能是点分布不均或共面")
        n+=1
            


Found 553 image pairs.
UAV: seu_uav_052409/00000.png - TIF: seu_tif_m300/29_0_6000.tif
start_x: 0, start_y: 6000
lu_geox: 668601.89603705, lu_geoy: 3548243.5491134794
(1062, 2)
Number of inliers: 299
(299, 2)
成功构建 3D 点集，共 299 个点
Yaw angle (degrees): 249.95
相机中心（世界坐标系）:
 [ 26.34762576 -51.97836473 117.42572833]
相机中心utm:：3548191.570748751, 668628.2436628101
UAV: seu_uav_052409/00001.png - TIF: seu_tif_m300/29_0_6000.tif
start_x: 0, start_y: 6000
lu_geox: 668601.89603705, lu_geoy: 3548243.5491134794
(1035, 2)
Number of inliers: 267
(267, 2)
成功构建 3D 点集，共 267 个点
Yaw angle (degrees): 250.18
相机中心（世界坐标系）:
 [ 26.50653486 -52.71370908 117.32363474]
相机中心utm:：3548190.835404401, 668628.4025719125
UAV: seu_uav_052409/00002.png - TIF: seu_tif_m300/29_0_6000.tif
start_x: 0, start_y: 6000
lu_geox: 668601.89603705, lu_geoy: 3548243.5491134794
(1061, 2)
Number of inliers: 308
(308, 2)
成功构建 3D 点集，共 308 个点
Yaw angle (degrees): 253.14
相机中心（世界坐标系）:
 [ 26.10802144 -53.26569719 116.93496345]
相机中心utm:：3548190.2

In [34]:

pairs = read_pairs(pairs_path)
print(f"Found {len(pairs)} image pairs.")
start_x, start_y = 0, 0
x_in_map, y_in_map = 0, 0
x_origin, y_origin = 0, 0
start = time.time()
n = 0
with open(loc_path, 'w') as loc_file:
    for img_uav, img_tif in pairs:
        print(f"UAV: {img_uav} - TIF: {img_tif}")
        kp1, kp2  = get_keypoints(features_path, img_uav), get_keypoints(features_path, img_tif)
        matches,scores = get_matches(matches_path, img_uav, img_tif)
        print(matches.shape)
        pts1 = kp1[matches[:,0]]
        pts2 = kp2[matches[:,1]]
        
        H, _ = cv2.findHomography(pts1, pts2, cv2.RANSAC, 1.0)
        print(H)
        if H is not None:
            h_uav, w_uav = 1080, 1920
            center_uav = np.array([[w_uav / 2, h_uav / 2]], dtype=np.float32)  
            noise1 = np.random.normal(-3, 3)  # 添加噪声
            noise2 = np.random.normal(-3, 3)  # 添加噪声
            center_uav[0][0] = center_uav[0][0]+noise1# 形状为 (1, 2)
            center_uav[0][1] = center_uav[0][1]+noise2# 形状为 (1, 2)
            # 将中心点坐标转换为齐次坐标
            center_uav_homogeneous = np.array([center_uav[0][0], center_uav[0][1], 1.0])  # 形状为 (3,)
            # 通过单应性矩阵进行变换
            center_tif_homogeneous = np.dot(H, center_uav_homogeneous)  # 形状为 (3,)
            # 归一化
            center_tif = center_tif_homogeneous[:2] / center_tif_homogeneous[2]  # 形状为 (2,)
            angle = extract_rotation_angle(H)
            print(f"旋转角度 (度): {angle}")
            # 打印结果
            print(f"无人机图像中心点在tif的位置：{center_tif}")
            pixel_x, pixel_y = center_tif
            tif_name = os.path.basename(img_tif)
            match = re.match(r"(\d+)_(\d+)_(\d+).tif", tif_name)
            if match:
                start_x = match.group(2)
                start_y = match.group(3)
                x_in_map = int(start_x) + pixel_x
                y_in_map = int(start_y) + pixel_y
                print(f"无人机图像中心点在地图上的位置：{x_in_map},{y_in_map}")
            try:
                lon, lat, x_geo, y_geo = pixel_to_geo_coordinates(x_in_map, y_in_map, geotransform,source_epsg=32650)
                print(f"无人机图像中心点的经纬度：{lat}, {lon}")
                if n == 0:
                    x_origin, y_origin = x_geo, y_geo
                loc_file.write(f"{img_uav} {lon:.8f} {lat:.8f} {x_in_map:.8f} {y_in_map:.8f} {x_geo:.8f} {y_geo:.8f} {x_geo-x_origin:.10f} {y_origin-y_geo:.10f} {angle:.8f}\n")
                n+=1
            except Exception as e:
                print(f"无法计算无人机图像中心点的经纬度：{e}")
end = time.time()
print(f"Time: {end-start:.3f}s")

Found 553 image pairs.
UAV: seu_uav_052409/00000.png - TIF: seu_tif_m300/29_0_6000.tif
(1062, 2)
[[ 1.72089052e+00  1.52675668e-01 -1.08907689e+03]
 [-1.04139019e-01  1.90319343e+00  3.83227353e+02]
 [-2.43456143e-05  1.19561136e-04  1.00000000e+00]]
旋转角度 (度): -4.053394409791281
无人机图像中心点在tif的位置：[ 614.40333884 1261.49576074]
无人机图像中心点在地图上的位置：614.4033388406563,7261.4957607405795
无人机图像中心点的经纬度：32.057458253420165, 118.78620318055364
UAV: seu_uav_052409/00001.png - TIF: seu_tif_m300/29_0_6000.tif
(1035, 2)
[[ 1.63871211e+00  1.20602551e-01 -1.02143164e+03]
 [-1.40984355e-01  1.76562994e+00  4.32791317e+02]
 [-5.17903553e-05  7.91830416e-05  1.00000000e+00]]
旋转角度 (度): -4.393927571575408
无人机图像中心点在tif的位置：[ 611.0134663  1256.66449627]
无人机图像中心点在地图上的位置：611.0134662974815,7256.6644962671835
无人机图像中心点的经纬度：32.05745977827963, 118.7862019679347
UAV: seu_uav_052409/00002.png - TIF: seu_tif_m300/29_0_6000.tif
(1061, 2)
[[ 1.70699780e+00  1.04826466e-01 -1.05369047e+03]
 [-1.30545731e-01  1.86766067e+00  3.9

# 生成地图轨迹

In [29]:
import cv2
from my_pkg.tools import parse_keyframe_file, draw_keyframe_trajectory, plot_traj_tif
points_traj = []
with open("/home/lty/outputs/seu0524/009/loc_elevpnp.txt", 'r') as loc_file:
    for line in loc_file:
        parts = line.strip().split()
        if len(parts) < 4:
            continue
        x_in_map = float(parts[3])*0.2   # 调整横坐标
        y_in_map = float(parts[4])*0.2  # 调整纵坐标
        print(f"{parts[0]}: {x_in_map}, {y_in_map}")
        points_traj.append((x_in_map, y_in_map))

    # 将点转换为整数坐标（像素坐标）
points_traj = [(int(x), int(y)) for x, y in points_traj]

# plot_traj_tif(
#     map_image_path="/home/lty/outputs/seu0524/009/gt_elevation.png",
#     loc_file_path=loc_path,
#     output_image_path=output_dir/"gt_elevation_H.png",
#     scale_factor=0.2,
# )

# 绘制关键帧的单独景象匹配结果
map = cv2.imread("/home/lty/outputs/seu0524/009/gt.png", cv2.IMREAD_COLOR)
keyframe_mapping = parse_keyframe_file("/home/lty/code/ORB_SLAM3_detailed_comments/KeyFrameId.txt")
map_with_traj = draw_keyframe_trajectory(map, points_traj, keyframe_mapping)
cv2.imwrite(output_dir/"gt_elevpnp.png", map_with_traj)

seu_uav_052409/00000.png: 152.298414798, 1500.45297531
seu_uav_052409/00001.png: 153.21696452199998, 1504.703520684
seu_uav_052409/00002.png: 150.91341871, 1507.894203404
seu_uav_052409/00003.png: 160.627562186, 1503.277355488
seu_uav_052409/00004.png: 153.801050354, 1505.5453423180002
seu_uav_052409/00005.png: 148.856026028, 1502.003861716
seu_uav_052409/00006.png: 157.01581248800002, 1499.79817783
seu_uav_052409/00007.png: 161.19852621400003, 1499.590773074
seu_uav_052409/00008.png: 152.22247451200002, 1507.5745313520001
seu_uav_052409/00009.png: 150.932310464, 1504.4181961660001
seu_uav_052409/00010.png: 155.058707038, 1506.329153146
seu_uav_052409/00011.png: 161.54644345600002, 1501.083587148
seu_uav_052409/00012.png: 161.084760608, 1499.777861492
seu_uav_052409/00013.png: 151.689029948, 1505.2423372560002
seu_uav_052409/00014.png: 153.96570231200002, 1504.591833664
seu_uav_052409/00015.png: 166.428651688, 1500.0506412500001
seu_uav_052409/00016.png: 146.396101888, 1507.08836221
se

True

slam traj

In [30]:
#fusion traj
import cv2
from my_pkg.tools import parse_keyframe_file, draw_slam_keyframe_traj, draw_fusion_keyframe_traj
slam_traj_path = "/home/lty/paper/results/052409/proposed.txt"
def geoXY_to_pixelXY(geo_x, geo_y, geotransform):
    x = int((geo_x - geotransform[0]) / geotransform[1])
    y = int((geo_y - geotransform[3]) / geotransform[5])
    return x, y

slam_traj = []
with open(slam_traj_path, 'r') as f:
    lines = f.readlines()
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 2:
            continue
        Pixelx, Pixely = geoXY_to_pixelXY(float(parts[0]), float(parts[1]), geotransform)
        #print(f"{parts[0]}: {Pixelx}, {Pixely}")
        x_in_map = Pixelx*0.2  # 调整横坐标  seu need *0.2
        y_in_map = Pixely*0.2
        slam_traj.append((x_in_map, y_in_map))
        
slam_traj = [(int(x), int(y)) for x, y in slam_traj]   
map = cv2.imread(output_dir/"gt_elevpnp.png", cv2.IMREAD_COLOR)
map_with_traj = draw_fusion_keyframe_traj(map, slam_traj)
cv2.imwrite(output_dir/"gt_elevpnp_proposed.png", map_with_traj)

True

In [31]:
#slam traj
import cv2
from my_pkg.tools import parse_keyframe_file, draw_slam_keyframe_traj
slam_traj_path = "/home/lty/paper/results/052409/slam.txt"
def geoXY_to_pixelXY(geo_x, geo_y, geotransform):
    x = int((geo_x - geotransform[0]) / geotransform[1])
    y = int((geo_y - geotransform[3]) / geotransform[5])
    return x, y

slam_traj = []
with open(slam_traj_path, 'r') as f:
    lines = f.readlines()
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 2:
            continue
        Pixelx, Pixely = geoXY_to_pixelXY(float(parts[0]), float(parts[1]), geotransform)
        #print(f"{parts[0]}: {Pixelx}, {Pixely}")
        x_in_map = Pixelx*0.2 # 调整横坐标  seu need *0.2
        y_in_map = Pixely *0.2
        slam_traj.append((x_in_map, y_in_map))
        
slam_traj = [(int(x), int(y)) for x, y in slam_traj]   
map = cv2.imread(output_dir/"gt_elevpnp_proposed.png", cv2.IMREAD_COLOR)
map_with_traj = draw_slam_keyframe_traj(map, slam_traj)
cv2.imwrite(output_dir/"compare0603.png", map_with_traj)

True

In [ ]:
# 打开 HDF5 文件
with h5py.File(features_path, 'r') as f:
    # 获取所有数据集的名称
    print("文件中的数据集和分组结构：")
    def print_structure(name, obj):
        """递归打印 HDF5 文件的层次结构"""
        if isinstance(obj, h5py.Group):
            print(f"Group: {name}")
        elif isinstance(obj, h5py.Dataset):
            print(f"Dataset: {name} - Shape: {obj.shape} - Type: {obj.dtype}")
    f.visititems(print_structure)

    # 示例：读取一个特定图像的特征
    example_image = "seu_uav/DJI_0218.JPG"  # 替换为您的图像名称
    if example_image in f:
        group = f[example_image]
        print(f"\n特定图像 '{example_image}' 的内容:")
        for key in group.keys():
            data = group[key][:]
            print(f"{key}: {data.shape} - {data.dtype}")
    else:
        print(f"图像 '{example_image}' 不存在于 HDF5 文件中。")